# Bloque 1 — Demo: Técnicas de Big Data

Este notebook acompaña la presentación del Bloque 1 y es el **notebook de referencia profundo**
para las técnicas de análisis sin IA de todo el curso: análisis de red (grafos, centralidad,
comunidades), TF-IDF y series temporales. Los casos de bloque 5 (CardingForums, HackingForums,
IronMarch) van a referenciar este notebook en vez de re-explicar cada técnica desde cero —
aquí es donde se ve el "cómo funciona" con calma; allá se aplica a cada foro concreto.

Usamos el dataset de **Carding Forums** como ejemplo porque es pequeño y limpio — perfecto para demos.

> **Prerrequisito**: haber ejecutado `CardingForums/01_ingenieria_datos.ipynb` para tener los archivos `.parquet` limpios.

## Imports

Cargamos las librerías que vamos a usar.

- **pandas**: para manipular tablas de datos (leer archivos, filtrar filas, agrupar valores)
- **networkx**: para construir y analizar grafos (redes de usuarios)
- **plotly**: para visualizaciones interactivas
- **community** (paquete `python-louvain`): para detectar comunidades con el algoritmo de Louvain — si no está instalado, más abajo usamos una alternativa de NetworkX puro sin dependencias extra
- **sklearn.feature_extraction.text.TfidfVectorizer**: para calcular TF-IDF sobre el texto de los posts

In [ ]:
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
from collections import Counter

# Intentamos importar python-louvain (paquete se llama 'community' al importarlo).
# Si no está instalado, usamos más abajo la alternativa de NetworkX puro
# (nx.community.greedy_modularity_communities) — mismo patrón try/except que
# usa el notebook de CardingForums (bloque5_cardingforums/02_analisis_estructural.ipynb).
try:
    import community as community_louvain
    HAS_LOUVAIN = True
except ImportError:
    HAS_LOUVAIN = False
    print('python-louvain no está instalado — usaremos nx.community.greedy_modularity_communities como alternativa.')

from sklearn.feature_extraction.text import TfidfVectorizer

DATA_DIR = Path('../results')
print('Directorio de datos:', DATA_DIR.resolve())

## 1. Cargar los datos limpios

Cargamos los archivos `.parquet` generados en el notebook de ingeniería de datos.

Un archivo **Parquet** es un formato de almacenamiento columnar muy eficiente — mucho más rápido de cargar que un CSV y ocupa menos espacio en disco. Pandas lo lee directamente con `pd.read_parquet()`.

In [ ]:
posts = pd.read_parquet(DATA_DIR / '01_posts_clean.parquet')
users = pd.read_parquet(DATA_DIR / '01_users_clean.parquet')

print(f'Posts cargados:   {len(posts):,}')
print(f'Usuarios cargados: {len(users):,}')
posts.head(3)

## 2. Análisis de red — ¿quién habla con quién?

### ¿Qué es un grafo de co-participación?

Dos usuarios están conectados si ambos han posteado en el mismo hilo de discusión.
La idea es que participar en el mismo hilo implica algún nivel de interacción o conocimiento mutuo.

Construimos el grafo así:
1. Para cada hilo, obtenemos la lista de usuarios que postearon en él
2. Conectamos todos los pares de usuarios de esa lista entre sí
3. Repetimos para todos los hilos

El resultado es un **grafo no dirigido** (si A está conectado con B, B también lo está con A —
no hay "sentido" en la relación, porque co-participar es simétrico) donde cada nodo es un usuario
y cada arista representa co-participación en al menos un hilo.

Cada arista además tiene un **peso** (`weight`): cuántos hilos distintos comparten esos dos usuarios.
Dos usuarios que sólo coincidieron una vez, por pura casualidad en un hilo masivo, tienen una arista
débil (peso 1); dos usuarios que siempre postean juntos tienen una arista fuerte (peso alto). Ese
peso es justo lo que usa Louvain más adelante para decidir qué nodos "pertenecen" a la misma comunidad.

Limitamos a los primeros 5.000 hilos para que el grafo sea manejable en RAM.

> **Para quien no tiene experiencia programando:** el código de abajo usa un `for` ("por cada elemento, repite este bloque") anidado dentro de otro `for` — es decir, un bucle dentro de otro bucle, para comparar cada usuario del hilo contra cada otro usuario del mismo hilo. También usa `.groupby(...)`, que es la forma que tiene pandas de decir "agrupa todas las filas que compartan el mismo valor en esta columna, y déjame procesar cada grupo por separado".

In [ ]:
thread_col = 'topic_id' if 'topic_id' in posts.columns else 'threadid'
user_col   = 'userid'

posts = posts.dropna(subset=[thread_col, user_col])

# Tomar los primeros 5000 hilos únicos
top_threads = posts[thread_col].value_counts().head(5000).index
df = posts[posts[thread_col].isin(top_threads)]

# Construir el grafo
G = nx.Graph()

# .groupby(thread_col) agrupa todas las filas (posts) que tienen el mismo hilo.
# 'thread_id' es el valor del hilo, 'group' es la sub-tabla de posts de ese hilo.
for thread_id, group in df.groupby(thread_col):
    # .unique() quita usuarios repetidos dentro del mismo hilo; .tolist() convierte
    # el resultado en una lista normal de Python para poder recorrerla con índices.
    users_in_thread = group[user_col].unique().tolist()

    # Bucle doble (uno dentro de otro): 'i' recorre cada usuario del hilo, y
    # 'j' recorre los usuarios que vienen DESPUÉS de 'i' en la misma lista.
    # Así generamos cada pareja de usuarios una sola vez, sin repetir ni comparar
    # un usuario consigo mismo.
    for i in range(len(users_in_thread)):
        for j in range(i + 1, len(users_in_thread)):
            u, v = str(users_in_thread[i]), str(users_in_thread[j])
            if G.has_edge(u, v):
                # G[u][v] accede al diccionario de atributos de la arista entre u y v;
                # ['weight'] es la clave que guarda cuántos hilos comparten.
                G[u][v]['weight'] += 1  # más hilos compartidos = arista más fuerte
            else:
                G.add_edge(u, v, weight=1)

print(f'Nodos (usuarios): {G.number_of_nodes():,}')
print(f'Aristas (co-participaciones): {G.number_of_edges():,}')

### Métricas de centralidad

**Degree centrality**: ¿con cuántos usuarios distintos ha coincidido este usuario? Un degree alto significa que el usuario es muy activo y alcanza a muchos otros. Es la métrica de "popularidad" más simple: solo mira conexiones directas, sin importar la forma general de la red.

**Betweenness centrality**: ¿cuántas rutas cortas entre otros pares de usuarios pasan por este nodo? Imagina que quieres ir de un usuario A a un usuario C dentro del grafo — la ruta más corta probablemente atraviese ciertos nodos intermedios. Un valor de betweenness alto indica que este usuario actúa de **puente**: si lo eliminas del grafo, grupos que antes estaban conectados quedarían separados.

La diferencia práctica importa: un usuario puede tener degree bajo (pocas conexiones directas) y aun así betweenness alto, si esas pocas conexiones son justo las que unen a dos subcomunidades que si no, no hablarían entre sí. Ese perfil — pocas conexiones pero estratégicas — es exactamente el de un intermediario o "broker": alguien que conecta partes de la red que de otro modo estarían aisladas. Estos son los actores más interesantes para inteligencia, muchas veces más que el usuario más "popular" por degree.

In [ ]:
# Trabajamos sobre el componente gigante (el subgrafo más grande)
giant = G.subgraph(max(nx.connected_components(G), key=len)).copy()
print(f'Componente gigante: {giant.number_of_nodes():,} nodos ({giant.number_of_nodes()/G.number_of_nodes()*100:.1f}% del total)')

degree     = nx.degree_centrality(giant)
# Betweenness sobre muestra de 500 nodos para ir rápido
betweenness = nx.betweenness_centrality(giant, k=min(500, giant.number_of_nodes()), normalized=True)

# dict(zip(a, b)) empareja cada elemento de 'a' con el elemento en la misma posición
# de 'b', y arma un diccionario {id_usuario: nombre_usuario}. Es la forma rápida de
# pandas/Python para construir una tabla de traducción id -> nombre.
uid_to_name = dict(zip(users['userid'].astype(str), users.get('username', users.get('name', users.index.astype(str)))))

# sorted(..., key=lambda x: x[1], reverse=True) ordena la lista de pares (usuario, valor)
# de mayor a menor según el segundo elemento de cada par (el valor de betweenness).
# 'lambda x: x[1]' es una función corta y anónima que dice "dame el segundo elemento de x".
# [:10] se queda solo con los primeros 10 después de ordenar.
top_bw = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:10]
print('\nTop 10 brokers (betweenness):')
for uid, bw in top_bw:
    # .get(uid, uid) busca uid en el diccionario; si no lo encuentra, usa el propio
    # uid como valor por defecto en vez de fallar con un error.
    name = uid_to_name.get(uid, uid)
    print(f'  {name:<25} {bw:.4f}')

### Detección de comunidades — algoritmo de Louvain

En una red existen grupos de nodos que están más densamente conectados entre sí que con el resto
de la red. A estos grupos los llamamos **comunidades**.

**Louvain** es un algoritmo que encuentra estas comunidades optimizando la **modularidad**: una
medida de qué tan bien separadas están las comunidades entre sí (modularidad más alta = comunidades
más nítidas, más internamente conectadas que hacia afuera). No necesitamos decirle de antemano
cuántas comunidades buscar — Louvain las descubre solo, fusionando nodos vecinos de forma iterativa
y local (en cada paso, agrupa el par de nodos/grupos cuya fusión más mejora la modularidad global)
hasta que ya no hay ninguna fusión que la mejore.

A diferencia de un algoritmo como k-means, Louvain no asume que los nodos viven en un espacio de
coordenadas donde "distancia" tiene sentido — trabaja directamente sobre las conexiones (y sus
pesos) del grafo. Por eso es la elección natural cuando lo que tenemos es una red de quién-conecta-
con-quién, y no un conjunto de puntos en un espacio.

> **Para quien no tiene experiencia programando:** el bloque de código de abajo usa `try/except`,
> una forma de decir "intenta hacer esto; si falla por un error específico (`ImportError`, que
> significa 'no encontré esta librería instalada'), haz esto otro en su lugar". Así el notebook
> funciona tanto si tienes `python-louvain` instalado como si no — el mismo patrón que usa el
> notebook de CardingForums (`bloque5_cardingforums/02_analisis_estructural.ipynb`, celda `f2a3b4c5`).

In [ ]:
# Mismo patrón try/except que el notebook de CardingForums (celda f2a3b4c5):
# intentamos python-louvain y, si no está disponible, caemos en la alternativa
# de NetworkX puro (más lenta, pero sin dependencias extra).
if HAS_LOUVAIN:
    partition = community_louvain.best_partition(giant, weight='weight', random_state=42)
    n_communities = len(set(partition.values()))
    modularity = community_louvain.modularity(partition, giant)
else:
    communities = nx.community.greedy_modularity_communities(giant)
    # Convertimos la lista de comunidades (cada una es un conjunto de nodos) en el
    # mismo formato que best_partition: un diccionario {nodo: id_de_comunidad}.
    partition = {}
    for i, comm in enumerate(communities):
        for node in comm:
            partition[node] = i
    n_communities = len(communities)
    modularity = nx.community.modularity(giant, communities)

print(f'Comunidades detectadas: {n_communities}')
print(f'Modularidad: {modularity:.3f}  (> 0.3 indica estructura de comunidad significativa)')

# Counter(partition.values()) cuenta cuántos nodos cayeron en cada id de comunidad —
# la misma utilidad que usa el notebook de CardingForums para reportar tamaños.
community_sizes = Counter(partition.values())
size_series = pd.Series(community_sizes).sort_values(ascending=False)
print('\nTamaño de comunidades (top 5):')
print(size_series.head(5).to_string())

### Visualización interactiva — top 150 nodos

Visualizamos los 150 nodos con mayor degree. El **color** del nodo representa su comunidad de
Louvain (cada color = una comunidad distinta detectada arriba) y el **tamaño** representa su
degree (más conexiones = nodo más grande).

Pasa el ratón sobre un nodo para ver el nombre del usuario.

In [ ]:
import numpy as np

# Esto es una "list comprehension": una forma compacta de escribir
# "por cada uid en top150, quédate solo con uid" en una sola línea,
# en vez de escribir un for tradicional que va añadiendo elementos a una lista vacía.
# sorted(..., key=lambda x: x[1], reverse=True) ordena los pares (usuario, degree)
# de mayor a menor degree, igual que hicimos antes con betweenness.
top150 = sorted(degree.items(), key=lambda x: x[1], reverse=True)[:150]
top150_ids = [uid for uid, _ in top150]
sub = giant.subgraph(top150_ids).copy()

# Layout del grafo
pos = nx.spring_layout(sub, seed=42, k=1.5)

edge_x, edge_y = [], []
for u, v in sub.edges():
    x0, y0 = pos[u]; x1, y1 = pos[v]
    edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

# Estas cuatro líneas son también list comprehensions: recorren 'sub.nodes()'
# (todos los nodos del subgrafo) y, para cada nodo 'n', calculan un valor
# (su posición x, su posición y, su nombre, su tamaño) para construir una lista.
node_x = [pos[n][0] for n in sub.nodes()]
node_y = [pos[n][1] for n in sub.nodes()]
node_names = [uid_to_name.get(n, n) for n in sub.nodes()]
# Coloreamos cada nodo por su comunidad de Louvain (partition.get devuelve -1 si un
# nodo no tuviera comunidad asignada, aunque no debería pasar aquí) y dimensionamos
# por degree: comunidad = color, actividad relativa = tamaño.
node_colors = [partition.get(n, -1) for n in sub.nodes()]
node_sizes  = [degree.get(n, 0) * 300 + 5 for n in sub.nodes()]

fig = go.Figure()
fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines',
                          line=dict(width=0.3, color='#888'), hoverinfo='none'))
fig.add_trace(go.Scatter(x=node_x, y=node_y, mode='markers+text',
                          marker=dict(size=node_sizes, color=node_colors, colorscale='Rainbow',
                                      showscale=False),
                          text=node_names, textposition='top center',
                          hovertemplate='<b>%{text}</b><extra></extra>'))
fig.update_layout(title='Red de co-participación — top 150 usuarios (color = comunidad Louvain)',
                  showlegend=False, height=700,
                  xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                  yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
fig.show()

## 3. Análisis temporal — ¿cuándo estaba activo el foro?

Los patrones temporales revelan mucho sobre una comunidad:
- **Ciclo de vida**: cuándo creció, cuándo murió
- **Horario de actividad**: inferencia de timezone (independientemente de lo que declare el usuario)
- **Eventos externos**: picos de actividad correlacionan con eventos reales

Recordatorio importante: filtramos posts con fechas anteriores a 2000 porque algunos registros tienen timestamp=0 (corrupto), que pandas interpreta como 1970-01-01.

In [ ]:
date_col = 'post_date' if 'post_date' in posts.columns else 'dateline'

df_time = posts.copy()
# pd.to_datetime convierte la columna de fecha (que puede venir como texto o número)
# en un tipo de fecha real que pandas entiende. '.dt' es el accesor que permite
# leer partes de una fecha (año, hora, etc.) de toda la columna a la vez.
df_time['date'] = pd.to_datetime(df_time[date_col], errors='coerce', utc=True)
df_time = df_time[df_time['date'].dt.year >= 2000]  # Filtrar epoch-0

# Actividad mensual
# .set_index('date') pone la columna de fecha como índice de la tabla.
# .resample('ME') agrupa las filas por mes (Month End) — es como un .groupby
# pero especializado en fechas — y .size() cuenta cuántas filas cayeron en cada mes.
monthly = df_time.set_index('date').resample('ME').size().reset_index()
monthly.columns = ['mes', 'posts']

fig = px.bar(monthly, x='mes', y='posts',
             title='Actividad mensual del foro',
             labels={'mes': 'Mes', 'posts': 'Posts'})
fig.show()

In [ ]:
# Distribución horaria — inferencia de timezone
df_time['hora'] = df_time['date'].dt.hour
# .groupby('hora').size() agrupa todos los posts por hora del día y cuenta
# cuántos hay en cada hora — el mismo patrón groupby que usamos antes con hilos.
hourly = df_time.groupby('hora').size().reset_index(name='posts')

fig = px.bar(hourly, x='hora', y='posts',
             title='Distribución horaria (UTC) — pico = timezone mayoritaria',
             labels={'hora': 'Hora UTC', 'posts': 'Posts'})
fig.show()

# .idxmax() devuelve la posición (índice) de la fila con el valor máximo de 'posts';
# .loc[...] usa ese índice para leer la columna 'hora' de esa fila concreta.
peak_hour = hourly.loc[hourly['posts'].idxmax(), 'hora']
print(f'Pico de actividad: {peak_hour}h UTC')

## 4. TF-IDF — ¿de qué habla cada foro?

### ¿Qué es TF-IDF?

**TF-IDF** significa *Term Frequency – Inverse Document Frequency* (frecuencia de término –
frecuencia inversa de documento). Es una medida de qué tan **característico** es un término
para un documento, comparado con el resto de los documentos.

Se calcula multiplicando dos partes:

- **TF (frecuencia de término)**: qué tan seguido aparece una palabra *dentro de este documento*.
  Si "dump" aparece 200 veces en el foro de carding, su TF ahí es alto.
- **IDF (frecuencia inversa de documento)**: qué tan *rara* es esa palabra en el conjunto completo
  de documentos. Palabras como "el", "de" o "the" aparecen en casi todos los documentos por igual,
  así que no distinguen nada — su IDF es bajo. Una palabra que solo aparece en uno o dos foros
  tiene IDF alto.

**TF-IDF alto = la palabra es frecuente aquí, pero rara en los demás** → es una palabra
característica y discriminante de este documento en particular. Una palabra con TF alto pero
IDF bajo (muy común en todos los foros) termina con TF-IDF bajo — TF-IDF filtra automáticamente
las palabras "de relleno" sin que tengamos que armar una lista manual de stopwords específica
del dominio.

En este notebook tratamos cada **foro completo** como un documento (una versión simplificada;
en el caso de CardingForums, `bloque5_cardingforums/02_analisis_estructural.ipynb` §7 hace lo
mismo pero a nivel de **subforo**, para distinguir tutoriales de dumps de cashout, por ejemplo).

> **Para quien no tiene experiencia programando:** `TfidfVectorizer` de scikit-learn hace todo
> el cálculo de TF e IDF por nosotros — solo le pasamos una lista de textos y nos devuelve una
> matriz numérica donde cada fila es un documento y cada columna es una palabra del vocabulario.

In [ ]:
# Agrupamos todo el texto de cada foro en un único "documento" por foro.
forum_col = 'forum' if 'forum' in posts.columns else None
text_col  = 'text_clean' if 'text_clean' in posts.columns else None

if forum_col and text_col:
    # .groupby(forum_col)[text_col] agrupa el texto por foro; .apply(...) une todos
    # los posts de cada foro en un solo string largo (uno por foro).
    forum_corpus = (
        posts
        .groupby(forum_col)[text_col]
        .apply(lambda textos: ' '.join(textos.dropna()))
    )

    vectorizer = TfidfVectorizer(
        max_features=2000,
        min_df=1,
        stop_words='english',   # elimina palabras vacías en inglés ('the', 'and', ...)
        ngram_range=(1, 2),     # incluye también pares de palabras ('credit card')
    )
    tfidf_matrix = vectorizer.fit_transform(forum_corpus)
    feature_names = vectorizer.get_feature_names_out()

    print('Términos más característicos por foro (top 10 por TF-IDF):')
    print('=' * 60)
    for i, forum_name in enumerate(forum_corpus.index):
        # .toarray()[0] convierte la fila dispersa (sparse) de este foro en un array
        # normal; .argsort() ordena los índices de menor a mayor TF-IDF, y [-10:][::-1]
        # se queda con los 10 más altos, ya invertidos de mayor a menor.
        top_indices = tfidf_matrix[i].toarray()[0].argsort()[-10:][::-1]
        top_terms = [feature_names[j] for j in top_indices]
        print(f"\n[{forum_name}]")
        print('  ' + ', '.join(top_terms))
else:
    print("Este dataset no tiene columna de texto limpio ('text_clean') o de foro ('forum');")
    print('el ejemplo de TF-IDF requiere ambas para poder agrupar y vectorizar por foro.')